1. Read the Bronze table
The catalog name contains a hyphen, so it is wrapped in backticks. spark.read.table() reads the current table snapshot in batch mode, which is appropriate for exploring all records currently stored in Bronze.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    LongType,
    DoubleType,
    BooleanType,
    TimestampType,
)

spark.conf.set("spark.sql.session.timeZone", "UTC")

BRONZE_TABLE = "`telcopulse-dev`.bronze.network_usages_eventhub"

bronze_df = spark.read.table(BRONZE_TABLE)

print(f"Bronze table: {BRONZE_TABLE}")
print(f"Total Bronze records: {bronze_df.count():,}")
display(bronze_df)

2. Check the Bronze schema
Confirm the physical column names and types before parsing. This notebook expects the columns created by your Event Hubs Bronze pipeline: event_key, event_value, headers, topic, partition, offset, kafka_timestamp, timestampType, and ingest_at.

In [0]:
bronze_df.printSchema()

# Check whether any expected Bronze columns are missing.
expected_columns = {
    "event_key", "event_value", "headers", "topic", "partition",
    "offset", "kafka_timestamp", "timestampType", "ingest_at"
}
missing_columns = expected_columns - set(bronze_df.columns)

if missing_columns:
    raise ValueError(f"Missing expected Bronze columns: {sorted(missing_columns)}")

3. Define the JSON schema
The schema is based on the fields observed in your supplied event sample. subscription_id, plan_id, plan_name, and plan_type are nullable. _corrupt_record captures malformed JSON during permissive parsing.

In [0]:
EVENT_SCHEMA = StructType([
    StructField("event_id", StringType(), True),
    StructField("event_version", IntegerType(), True),
    StructField("source_system", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("sequence_number", LongType(), True),
    StructField("subscriber_id", StringType(), True),
    StructField("device_id", StringType(), True),
    StructField("subscription_id", StringType(), True),
    StructField("plan_id", StringType(), True),
    StructField("plan_name", StringType(), True),
    StructField("plan_type", StringType(), True),
    StructField("cell_site_id", StringType(), True),
    StructField("cell_site_city", StringType(), True),
    StructField("cell_site_region", StringType(), True),
    StructField("network_technology", StringType(), True),
    StructField("event_timestamp", TimestampType(), True),
    StructField("ingestion_timestamp", TimestampType(), True),
    StructField("download_mb", DoubleType(), True),
    StructField("upload_mb", DoubleType(), True),
    StructField("latency_ms", IntegerType(), True),
    StructField("packet_loss_pct", DoubleType(), True),
    StructField("signal_strength_dbm", IntegerType(), True),
    StructField("duration_seconds", LongType(), True),
    StructField("roaming", BooleanType(), True),
    StructField("data_quality_hint", StringType(), True),
    StructField("_corrupt_record", StringType(), True),
])

print(f"JSON schema fields (including corrupt-record field): {len(EVENT_SCHEMA.fields)}")

4. Parse event_value
from_json() converts the JSON string into a typed struct called _payload. The original event_value remains available in bronze_df if you need to compare the parsed output with the raw JSON.

In [0]:
parsed_df = bronze_df.withColumn(
    "_payload",
    F.from_json(
        F.col("event_value"),
        EVENT_SCHEMA,
        {
            "mode": "PERMISSIVE",
            "columnNameOfCorruptRecord": "_corrupt_record",
        },
    ),
)

parsed_df = parsed_df.withColumn(
    "json_parse_status",
    F.when(
        F.col("_payload").isNull()
        | F.col("_payload._corrupt_record").isNotNull(),
        F.lit("INVALID"),
    ).otherwise(F.lit("VALID")),
)

display(
    parsed_df.select(
        "event_key", "topic", "partition", "offset",
        "json_parse_status", "_payload"
    )
)

5. Flatten the JSON and keep useful Kafka lineage
This view keeps the business event fields plus useful Kafka identifiers (topic, partition, offset, broker timestamp) and the Bronze ingestion timestamp. It does not write a Silver table.

In [0]:
exploration_df = parsed_df.select(
    # Kafka lineage
    F.col("event_key").alias("kafka_message_key"),
    F.col("topic").alias("kafka_topic"),
    F.col("partition").cast("integer").alias("kafka_partition"),
    F.col("offset").cast("long").alias("kafka_offset"),
    F.col("kafka_timestamp"),
    F.col("timestampType").alias("kafka_timestamp_type"),
    F.col("headers").alias("kafka_headers"),
    F.concat_ws(
        ":",
        F.col("topic"),
        F.col("partition").cast("string"),
        F.col("offset").cast("string"),
    ).alias("kafka_record_id"),

    # Event identity and common dimensions
    F.col("_payload.event_id").alias("event_id"),
    F.col("_payload.event_version").alias("event_version"),
    F.col("_payload.source_system").alias("source_system"),
    F.col("_payload.event_type").alias("event_type"),
    F.col("_payload.sequence_number").alias("sequence_number"),
    F.col("_payload.subscriber_id").alias("subscriber_id"),
    F.col("_payload.device_id").alias("device_id"),
    F.col("_payload.subscription_id").alias("subscription_id"),
    F.col("_payload.plan_id").alias("plan_id"),
    F.col("_payload.plan_name").alias("plan_name"),
    F.col("_payload.plan_type").alias("plan_type"),
    F.col("_payload.cell_site_id").alias("cell_site_id"),
    F.col("_payload.cell_site_city").alias("cell_site_city"),
    F.col("_payload.cell_site_region").alias("cell_site_region"),
    F.col("_payload.network_technology").alias("network_technology"),

    # Event timestamps
    F.col("_payload.event_timestamp").alias("event_timestamp"),
    F.col("_payload.ingestion_timestamp").alias("producer_ingestion_timestamp"),
    F.col("ingest_at").alias("bronze_ingest_at"),

    # Network metrics
    F.col("_payload.download_mb").alias("download_mb"),
    F.col("_payload.upload_mb").alias("upload_mb"),
    F.col("_payload.latency_ms").alias("latency_ms"),
    F.col("_payload.packet_loss_pct").alias("packet_loss_pct"),
    F.col("_payload.signal_strength_dbm").alias("signal_strength_dbm"),
    F.col("_payload.duration_seconds").alias("duration_seconds"),
    F.col("_payload.roaming").alias("roaming"),
    F.col("_payload.data_quality_hint").alias("data_quality_hint"),

    # Parsing result
    F.col("json_parse_status"),
)

print(f"Parsed records: {exploration_df.count():,}")
exploration_df.printSchema()
display(exploration_df)

This summary helps check the mix of DATA_SESSION, VOICE_CALL, and SMS events in the Bronze table.

In [0]:
display(
    exploration_df
    .groupBy("event_type")
    .count()
    .orderBy(F.desc("count"))
)

7. Check nullable fields
These fields were nullable in the sample payload. Review the counts using the full Bronze table before deciding whether any should be mandatory.

In [0]:
nullable_columns = [
    "subscription_id",
    "plan_id",
    "plan_name",
    "plan_type",
]

null_counts = exploration_df.agg(*[
    F.sum(F.col(c).isNull().cast("long")).alias(c)
    for c in nullable_columns
])

display(null_counts)

8. Inspect invalid JSON records
If this result is empty, no malformed JSON rows were found in the current Bronze snapshot. This does not check every business rule; it checks JSON parsing only.

In [0]:
invalid_json_df = parsed_df.filter(
    F.col("json_parse_status") == "INVALID"
).select(
    "topic", "partition", "offset", "kafka_timestamp", "event_value",
    F.col("_payload._corrupt_record").alias("corrupt_record"),
)

print(f"Invalid JSON records: {invalid_json_df.count():,}")
display(invalid_json_df)